# Resume Parser using BERT Transformer (GPU-Accelerated)

A high-performance resume parsing system that leverages BERT NER with CUDA GPU acceleration to extract comprehensive structured data from resumes in PDF, DOCX, and DOC formats.

**Pipeline Overview:**
1. Text extraction from multiple document formats (PDF/DOCX/DOC)
2. GPU-accelerated Named Entity Recognition using `dslim/bert-base-NER`
3. Multi-strategy information extraction (NER + regex + spaCy + heuristics)
4. Structured JSON output with all extracted fields

**Fields Extracted:**
- Contact: Name, Email, Phone, LinkedIn, GitHub, Portfolio, Address
- Education: Degree, Institution, Year, CGPA/Percentage, Specialization
- Work Experience: Company, Role, Duration, Location, Description, Type
- Skills: Technical Skills, Soft Skills, Tools, Frameworks
- Projects: Title, Description, Tech Stack, Links
- Certifications, Achievements, Languages, Objective/Summary, Hobbies

## Step 1: Install Required Libraries

In [1]:
# installing all the packages we need for this project
%pip install -q transformers torch python-docx PyPDF2 spacy
!python -m spacy download en_core_web_sm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 114.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 176.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## Step 2: Import Libraries and Configure GPU

In [2]:
import os
import re
import json
import warnings
from pathlib import Path
from collections import OrderedDict

warnings.filterwarnings("ignore")

# document format handlers
import docx
import PyPDF2

# NLP stuff
import spacy

# torch for GPU handling
import torch

# huggingface transformers - this is where the magic happens
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

# check if we have a GPU available, use it if we do
device = 0 if torch.cuda.is_available() else -1
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

print(f"[+] Libraries loaded")
print(f"[+] Compute Device: {device_name}")
print(f"[+] CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[+] GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

[+] Libraries loaded
[+] Compute Device: CPU
[+] CUDA Available: False


## Step 3: Document Text Extraction

Handles all three file formats - extracts raw text preserving structure as much as possible. Also grabs content from tables in docx files since many resumes use table layouts.

In [3]:
def extract_text_from_pdf(file_path):
    """reads each page of the pdf and concatenates all text"""
    text = ""
    with open(file_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            content = page.extract_text()
            if content:
                text += content + "\n"
    if not text.strip():
        raise ValueError(f"PDF appears to be scanned/image-only: {file_path}")
    return text.strip()


def extract_text_from_docx(file_path):
    """pulls text from paragraphs AND tables - many resumes use table layouts"""
    doc = docx.Document(file_path)
    chunks = []

    # grab paragraph text
    for para in doc.paragraphs:
        if para.text.strip():
            chunks.append(para.text)

    # grab table content too - resumes often use invisible tables for layout
    for table in doc.tables:
        for row in table.rows:
            row_text = []
            for cell in row.cells:
                if cell.text.strip() and cell.text.strip() not in row_text:
                    row_text.append(cell.text.strip())
            if row_text:
                chunks.append(" | ".join(row_text))

    text = "\n".join(chunks)
    if not text.strip():
        raise ValueError(f"DOCX file is empty: {file_path}")
    return text.strip()


def extract_text_from_doc(file_path):
    """handles legacy .doc format - tries docx parse first, then antiword fallback"""
    # a lot of .doc files are actually docx with wrong extension
    try:
        return extract_text_from_docx(file_path)
    except Exception:
        pass

    # try antiword if its installed on the system
    import subprocess
    try:
        result = subprocess.run(
            ["antiword", file_path],
            capture_output=True, text=True, check=True, timeout=30
        )
        if result.stdout.strip():
            return result.stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
        pass

    raise ValueError(f"Cannot read .doc file: {file_path}. Convert to .docx or .pdf")


def extract_text(file_path):
    """main entry point - figures out the format and delegates to the right extractor"""
    file_path = str(file_path)
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"No such file: {file_path}")

    ext = os.path.splitext(file_path)[1].lower()

    handlers = {
        ".pdf": extract_text_from_pdf,
        ".docx": extract_text_from_docx,
        ".doc": extract_text_from_doc
    }

    if ext not in handlers:
        raise ValueError(f"Format '{ext}' not supported. Use .pdf, .docx, or .doc")

    return handlers[ext](file_path)


print("[+] Text extraction functions ready")

[+] Text extraction functions ready


## Step 4: Load BERT NER Model (GPU-Accelerated)

Using `dslim/bert-base-NER` which is BERT-base fine-tuned on CoNLL-2003 dataset. This model recognizes Person, Organization, Location and Misc entities. We push the model to GPU if available for faster inference.

In [4]:
# using dslim/bert-base-NER - its one of the best open-source NER models
# trained on CoNLL-2003, recognizes: PER, ORG, LOC, MISC
model_name = "dslim/bert-base-NER"

print(f"[*] Loading model: {model_name}")
print(f"[*] Target device: {'CUDA GPU' if device == 0 else 'CPU'}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

# if GPU available, move model there for faster inference
if device == 0:
    model = model.to("cuda")
    print(f"[+] Model moved to GPU")

# build the NER pipeline
# aggregation_strategy="simple" merges subword tokens (e.g. "Micro" + "##soft" -> "Microsoft")
ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=device  # 0 for GPU, -1 for CPU
)

# spacy for backup NER and sentence splitting
nlp = spacy.load("en_core_web_sm")

print(f"[+] BERT NER pipeline ready (device={'gpu' if device==0 else 'cpu'})")
print(f"[+] spaCy en_core_web_sm loaded")
print(f"[+] Tokenizer max length: {tokenizer.model_max_length}")

[*] Loading model: dslim/bert-base-NER
[*] Target device: CPU


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[+] BERT NER pipeline ready (device=cpu)
[+] spaCy en_core_web_sm loaded
[+] Tokenizer max length: 512


## Step 5: Information Extraction Engine

This is where the heavy lifting happens. We combine three approaches:
1. **BERT NER** - identifies names, orgs, locations automatically
2. **Regex patterns** - catches structured stuff like emails, phones, dates
3. **Heuristic rules** - section boundary detection, context-aware parsing

Each section of the resume gets its own specialized extractor.

In [5]:
# ----------------------------------------------------------------
# SECTION BOUNDARY DETECTION
# ----------------------------------------------------------------
# resumes are organized into sections with headers like "Education", "Skills" etc.
# this function figures out where each section starts and ends

# these are basically all the possible section headings we might encounter
SECTION_PATTERNS = [
    r'education|academic|qualification|scholastic',
    r'experience|employment|work\s*history|professional\s*experience|career',
    r'skills|technical\s*skills|core\s*competencies|technologies|proficiency|expertise',
    r'certification|certifi|license|accreditation|credential',
    r'project|personal\s*project|academic\s*project|key\s*project',
    r'objective|summary|profile|about\s*me|career\s*objective|professional\s*summary',
    r'achievement|award|honor|accomplishment',
    r'language|linguistic',
    r'interest|hobby|hobbies|extracurricular|activities',
    r'reference|publication|research|patent',
    r'training|workshop|course|professional\s*development',
    r'volunteer|social|community',
    r'declaration|personal\s*info|personal\s*detail',
]


def get_section_text(text, target_keywords):
    """
    finds the section matching target_keywords and returns
    all text between that header and the next section header.

    the trick here is that section headers are usually short lines
    (under ~40 chars) that match our keyword patterns.
    """
    lines = text.split('\n')
    capturing = False
    captured = []

    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue

        # is this line a section header?
        is_any_header = False
        is_our_header = False

        # check against all known section patterns
        for pattern in SECTION_PATTERNS:
            # headers are typically short - if line is >50 chars its probably not a heading
            if len(stripped) < 50 and re.search(pattern, stripped, re.IGNORECASE):
                is_any_header = True
                break

        # check if its specifically the section we want
        for kw in target_keywords:
            if re.search(kw, stripped, re.IGNORECASE) and len(stripped) < 50:
                is_our_header = True
                break

        if is_our_header and not capturing:
            capturing = True  # start grabbing lines
            continue
        elif is_any_header and capturing:
            break  # we hit the next section, stop
        elif capturing:
            captured.append(stripped)

    return '\n'.join(captured)


# ----------------------------------------------------------------
# CONTACT INFO EXTRACTION
# ----------------------------------------------------------------
# pulls out: name, email, phone, linkedin, github, portfolio, address

def extract_contact_info(text, ner_results):
    """
    extracts all contact-related fields from the resume.
    uses NER for name detection, regex for everything else.
    """
    contact = {}

    # -- NAME --
    # first try: get it from BERT NER results (PER entities)
    person_entities = [
        e["word"].replace("##", "").strip()
        for e in ner_results
        if e["entity_group"] == "PER" and e["score"] > 0.8 and len(e["word"].strip()) > 2
    ]

    if person_entities:
        contact["name"] = person_entities[0]
    else:
        # fallback: spacy NER on the first few lines
        doc = nlp(text[:500])
        for ent in doc.ents:
            if ent.label_ == "PERSON" and len(ent.text) > 3:
                contact["name"] = ent.text
                break

    # last resort: first line of resume is often just the name
    if "name" not in contact:
        first_lines = [l.strip() for l in text.split('\n') if l.strip()]
        if first_lines:
            candidate = first_lines[0]
            # name should be short, no emails/urls/numbers
            if len(candidate) < 40 and not re.search(r'@|http|www|\d{4,}|resume|curriculum', candidate, re.IGNORECASE):
                contact["name"] = candidate

    # -- EMAIL --
    email_regex = r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}'
    emails = re.findall(email_regex, text)
    contact["email"] = emails[0] if emails else None

    # -- PHONE --
    # handles multiple formats: +91-XXXXX-XXXXX, (555) 123-4567, +1 555 123 4567, etc.
    phone_regexes = [
        r'\+?\d{1,3}[-.\s]?\(?\d{2,5}\)?[-.\s]?\d{3,5}[-.\s]?\d{3,5}',  # international
        r'\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}',  # US/Canada
        r'\+?\d{1,3}[-.\s]?\d{10}',  # continuous 10-digit with country code
        r'\d{5}[-.\s]?\d{5}',  # Indian style 5+5
    ]
    for regex in phone_regexes:
        matches = re.findall(regex, text)
        if matches:
            # filter out stuff thats clearly not a phone (like years, zip codes)
            for m in matches:
                digits_only = re.sub(r'\D', '', m)
                if 10 <= len(digits_only) <= 13:
                    contact["phone"] = m.strip()
                    break
            if "phone" in contact:
                break
    if "phone" not in contact:
        contact["phone"] = None

    # -- LINKEDIN --
    linkedin_regex = r'(?:https?://)?(?:www\.)?linkedin\.com/in/[\w\-./]+'
    match = re.search(linkedin_regex, text, re.IGNORECASE)
    contact["linkedin"] = match.group().rstrip('/. ') if match else None

    # -- GITHUB --
    github_regex = r'(?:https?://)?(?:www\.)?github\.com/[\w\-.]+'
    match = re.search(github_regex, text, re.IGNORECASE)
    contact["github"] = match.group().rstrip('/. ') if match else None

    # -- PORTFOLIO / WEBSITE --
    # look for any other URLs that aren't linkedin/github
    url_regex = r'https?://(?:www\.)?[\w\-]+\.[\w\-]+(?:/[\w\-./]*)?'
    all_urls = re.findall(url_regex, text)
    other_urls = [
        u for u in all_urls
        if 'linkedin' not in u.lower() and 'github' not in u.lower()
    ]
    contact["portfolio"] = other_urls[0] if other_urls else None

    # -- ADDRESS / LOCATION --
    # try to find city/state/country patterns
    loc_entities = [
        e["word"].replace("##", "").strip()
        for e in ner_results
        if e["entity_group"] == "LOC" and e["score"] > 0.8
    ]
    contact["location"] = loc_entities[0] if loc_entities else None

    return contact


# ----------------------------------------------------------------
# EDUCATION EXTRACTION
# ----------------------------------------------------------------

def extract_education(text):
    """
    parses the education section, looking for:
    - degree/program name
    - institution name
    - graduation year or duration
    - CGPA, GPA, or percentage
    - specialization/major
    """
    education = []

    edu_text = get_section_text(text, [r'education', r'academic', r'qualification', r'scholastic'])
    if not edu_text:
        return education

    lines = [l.strip() for l in edu_text.split('\n') if l.strip()]

    # patterns for detecting degree names
    degree_regex = r'(B\.?\s?Tech|B\.?\s?E\.?|B\.?\s?Sc|B\.?\s?S\.?|B\.?\s?A\.?|B\.?\s?Com|BCA|BBA|B\.?Arch|M\.?\s?Tech|M\.?\s?E\.?|M\.?\s?Sc|M\.?\s?S\.?|M\.?\s?A\.?|M\.?\s?Com|MCA|MBA|M\.?Phil|Ph\.?\s?D|Diploma|Bachelor|Master|Doctor|Associate|HSC|SSC|CBSE|ICSE|12th|10th|Intermediate|Higher\s*Secondary|Secondary|High\s*School)'

    # institution keywords
    inst_regex = r'(university|college|institute|school|academy|iit|nit|iiit|bits|vit|srm|manipal|amity)'

    current_edu = {}

    for line in lines:
        # does this line have a degree keyword?
        if re.search(degree_regex, line, re.IGNORECASE):
            # save previous entry if we had one
            if current_edu and any(k in current_edu for k in ["degree", "institution"]):
                education.append(current_edu)
            current_edu = {"degree": line}

        # institution name
        elif re.search(inst_regex, line, re.IGNORECASE):
            current_edu["institution"] = line

        # look for years
        years = re.findall(r'((?:19|20)\d{2})', line)
        if years:
            # if theres two years, its a duration like 2019-2023
            if len(years) >= 2:
                current_edu["start_year"] = years[0]
                current_edu["graduation_year"] = years[-1]
            elif "graduation_year" not in current_edu:
                current_edu["graduation_year"] = years[-1]

        # CGPA or GPA
        gpa = re.search(r'(?:CGPA|GPA|CPI|SPI)[\s:]*(\d+\.?\d*)\s*(?:/\s*(\d+))?', line, re.IGNORECASE)
        if gpa:
            current_edu["cgpa"] = gpa.group(1)
            if gpa.group(2):
                current_edu["cgpa_scale"] = gpa.group(2)

        # percentage
        pct = re.search(r'(\d{2,3}\.?\d*)\s*(%|percent)', line, re.IGNORECASE)
        if pct:
            current_edu["percentage"] = pct.group(1) + "%"

        # specialization/major/stream
        spec = re.search(r'(?:specializ|major|stream|branch|field)[\s:in]*(.+)', line, re.IGNORECASE)
        if spec:
            current_edu["specialization"] = spec.group(1).strip()

    if current_edu:
        education.append(current_edu)

    return education


# ----------------------------------------------------------------
# WORK EXPERIENCE EXTRACTION
# ----------------------------------------------------------------

def extract_work_experience(text):
    """
    parses work/professional experience:
    - company name
    - job title/position
    - duration (start - end)
    - location
    - description of responsibilities
    - employment type (full-time, intern, etc.)
    """
    experience = []

    exp_text = get_section_text(text, [
        r'experience', r'employment', r'work\s*history',
        r'professional\s*experience', r'career\s*history'
    ])
    if not exp_text:
        return experience

    lines = [l.strip() for l in exp_text.split('\n') if l.strip()]

    # bunch of date formats people use on resumes
    date_regexes = [
        # "Jan 2020 - Present", "January 2020 - Dec 2022"
        r'((?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:t(?:ember)?)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)[.\s]*\'?\d{2,4}\s*[-–to]+\s*(?:(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:t(?:ember)?)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)[.\s]*\'?\d{2,4}|[Pp]resent|[Cc]urrent|[Tt]ill\s*[Dd]ate|[Oo]ngoing))',
        # "2020 - 2023", "2021 - Present"
        r'((?:19|20)\d{2}\s*[-–to]+\s*(?:(?:19|20)\d{2}|[Pp]resent|[Cc]urrent|[Tt]ill\s*[Dd]ate|[Oo]ngoing))',
        # "06/2020 - 12/2022"
        r'(\d{1,2}/\d{4}\s*[-–to]+\s*(?:\d{1,2}/\d{4}|[Pp]resent|[Cc]urrent))',
    ]

    # words that indicate a job title
    title_words = r'(engineer|developer|programmer|manager|analyst|designer|architect|consultant|director|lead|senior|junior|intern|trainee|executive|associate|specialist|coordinator|administrator|scientist|researcher|officer|head|vp|president|founder|co-founder|cto|ceo|devops|qa|tester|full[\s-]?stack|front[\s-]?end|back[\s-]?end|data|cloud|sde|swe|mts)'

    # words that suggest a company name
    company_words = r'(inc\.?|corp\.?|ltd\.?|llc|pvt\.?|private|limited|company|technologies|solutions|software|services|systems|consulting|infosys|tcs|wipro|cognizant|accenture|capgemini|deloitte|google|amazon|microsoft|meta|apple|flipkart|ola|swiggy|paytm|razorpay|group|ventures|labs|digital|studios|media)'

    # employment type markers
    emp_type_regex = r'(?i)(full[\s-]?time|part[\s-]?time|contract|freelance|intern(?:ship)?|temporary|remote|hybrid|on[\s-]?site)'

    current_exp = {}
    desc_buffer = []

    for line in lines:
        # check for duration/dates
        date_found = None
        for regex in date_regexes:
            m = re.search(regex, line)
            if m:
                date_found = m.group().strip()
                break

        if date_found:
            # new experience entry - save the old one first
            if current_exp:
                if desc_buffer:
                    current_exp["description"] = " | ".join(desc_buffer)
                experience.append(current_exp)
                desc_buffer = []

            current_exp = {"duration": date_found}
            leftover = line.replace(date_found, "").strip(" -–|,/")
            if leftover:
                # whats left might be position or company
                if re.search(title_words, leftover, re.IGNORECASE):
                    current_exp["position"] = leftover
                else:
                    current_exp["company"] = leftover

        # company name detection
        elif re.search(company_words, line, re.IGNORECASE) and "company" not in current_exp:
            current_exp["company"] = line

        # job title detection
        elif re.search(title_words, line, re.IGNORECASE) and "position" not in current_exp:
            current_exp["position"] = line

        # employment type (full-time, intern etc)
        elif re.search(emp_type_regex, line):
            emp_match = re.search(emp_type_regex, line)
            current_exp["employment_type"] = emp_match.group().strip()
            # rest of line might be useful
            rest = re.sub(emp_type_regex, '', line).strip(" -–|,")
            if rest and len(rest) > 3:
                desc_buffer.append(rest)

        # anything else goes into description
        else:
            if current_exp:
                desc_buffer.append(line)

    # dont forget the last entry
    if current_exp:
        if desc_buffer:
            current_exp["description"] = " | ".join(desc_buffer)
        experience.append(current_exp)

    return experience


# ----------------------------------------------------------------
# SKILLS EXTRACTION
# ----------------------------------------------------------------

def extract_skills(text):
    """
    grabs skills from the skills section.
    splits them by common delimiters (commas, pipes, bullets etc.)
    also tries to categorize into technical vs tools vs frameworks
    """
    skills_text = get_section_text(text, [
        r'skill', r'technical\s*skill', r'core\s*competenc',
        r'technologies', r'proficiency', r'expertise', r'tech\s*stack'
    ])

    if not skills_text:
        return {"technical_skills": [], "tools": [], "frameworks": []}

    all_skills = []
    lines = [l.strip() for l in skills_text.split('\n') if l.strip()]

    for line in lines:
        # split on any common delimiter used in skill lists
        items = re.split(r'[,;|•·▪►■●○◆→★✓\-]|\band\b', line)
        for item in items:
            item = item.strip(' :–—\t()[]')
            # a valid skill should be 2-50 chars and not just numbers
            if item and 2 <= len(item) <= 50 and not re.match(r'^[\d\s]+$', item):
                # skip section header words that sneak in
                if not re.match(r'(?i)^(skill|technical|core|proficiency|competenc|tool|framework|language|technology)', item):
                    all_skills.append(item)

    # deduplicate (case insensitive) while keeping original casing
    seen = set()
    unique = []
    for s in all_skills:
        key = s.lower().strip()
        if key not in seen and len(key) > 1:
            seen.add(key)
            unique.append(s)

    # try to categorize - known tools, frameworks, etc.
    tools_keywords = {'git', 'docker', 'kubernetes', 'jenkins', 'jira', 'vs code', 'vscode', 'postman', 'figma', 'slack', 'trello', 'linux', 'aws', 'azure', 'gcp', 'heroku', 'netlify', 'vercel', 'terraform', 'ansible', 'vagrant', 'vim', 'eclipse', 'intellij'}
    framework_keywords = {'react', 'angular', 'vue', 'django', 'flask', 'spring', 'express', 'fastapi', 'next.js', 'nextjs', 'nuxt', 'svelte', 'tailwind', 'bootstrap', 'jquery', 'tensorflow', 'pytorch', 'keras', 'scikit', 'pandas', 'numpy', 'matplotlib', 'node.js', 'nodejs', '.net', 'laravel', 'rails'}

    categorized = {
        "technical_skills": [],
        "tools": [],
        "frameworks": []
    }

    for skill in unique:
        sl = skill.lower()
        if sl in tools_keywords or any(t in sl for t in tools_keywords):
            categorized["tools"].append(skill)
        elif sl in framework_keywords or any(f in sl for f in framework_keywords):
            categorized["frameworks"].append(skill)
        else:
            categorized["technical_skills"].append(skill)

    return categorized


# ----------------------------------------------------------------
# CERTIFICATIONS EXTRACTION
# ----------------------------------------------------------------

def extract_certifications(text):
    """gets certs/licenses/accreditations with their year if mentioned"""
    certifications = []

    cert_text = get_section_text(text, [r'certification', r'certifi', r'license', r'accreditation', r'credential'])
    if not cert_text:
        return certifications

    lines = [l.strip() for l in cert_text.split('\n') if l.strip()]

    for line in lines:
        if len(line) > 3:
            cert_entry = {"name": line}
            # try to extract year
            year_match = re.search(r'((?:19|20)\d{2})', line)
            if year_match:
                cert_entry["year"] = year_match.group(1)
                cert_entry["name"] = re.sub(r'\(?(?:19|20)\d{2}\)?', '', line).strip(' -–,.()')
            certifications.append(cert_entry)

    return certifications


# ----------------------------------------------------------------
# PROJECTS EXTRACTION
# ----------------------------------------------------------------

def extract_projects(text):
    """
    parses projects section - tries to identify:
    - project title
    - description/details
    - tech stack used
    - links (github, demo)
    """
    projects = []

    proj_text = get_section_text(text, [r'project', r'key\s*project', r'personal\s*project'])
    if not proj_text:
        return projects

    lines = [l.strip() for l in proj_text.split('\n') if l.strip()]

    current_proj = {}
    desc_lines = []

    for line in lines:
        # a project title is typically a short line that starts with uppercase
        # and doesn't start with a bullet point
        looks_like_title = (
            len(line) < 80 and
            line[0].isupper() and
            not line.startswith(('•', '-', '*', '●', '○', '→', '►', '◆')) and
            not re.match(r'^\d+[.)]\s', line) and
            not re.match(r'(?i)^(tech|tool|stack|built|developed|created|designed|implemented)', line)
        )

        if looks_like_title:
            # save previous project
            if current_proj:
                if desc_lines:
                    current_proj["description"] = " | ".join(desc_lines)
                projects.append(current_proj)
                desc_lines = []
            current_proj = {"title": line}
        else:
            # check if this line mentions tech stack
            if re.match(r'(?i)(tech|stack|tools?|built\s*(?:with|using)|technologies?\s*used)', line):
                tech_part = re.sub(r'(?i)^(tech\s*stack|tools?\s*used|built\s*(?:with|using)|technologies?\s*used)[\s:]*', '', line)
                current_proj["tech_stack"] = tech_part.strip()
            # check for links
            elif re.search(r'(?:github|demo|live|link|hosted|deploy)', line, re.IGNORECASE):
                urls = re.findall(r'https?://[\w\-./]+', line)
                if urls:
                    current_proj["link"] = urls[0]
                else:
                    desc_lines.append(line)
            else:
                desc_lines.append(line)

    # last one
    if current_proj:
        if desc_lines:
            current_proj["description"] = " | ".join(desc_lines)
        projects.append(current_proj)

    return projects


# ----------------------------------------------------------------
# OBJECTIVE / SUMMARY EXTRACTION
# ----------------------------------------------------------------

def extract_objective(text):
    """gets the career objective or professional summary if present"""
    obj_text = get_section_text(text, [r'objective', r'summary', r'profile', r'about\s*me', r'career\s*objective'])
    if obj_text:
        return obj_text.strip()
    return None


# ----------------------------------------------------------------
# ACHIEVEMENTS / AWARDS EXTRACTION
# ----------------------------------------------------------------

def extract_achievements(text):
    """parses achievements, awards, honors"""
    achievements = []

    ach_text = get_section_text(text, [r'achievement', r'award', r'honor', r'accomplishment'])
    if not ach_text:
        return achievements

    lines = [l.strip() for l in ach_text.split('\n') if l.strip()]
    for line in lines:
        if len(line) > 3:
            achievements.append(line)

    return achievements


# ----------------------------------------------------------------
# LANGUAGES EXTRACTION
# ----------------------------------------------------------------

def extract_languages(text):
    """gets spoken/known languages"""
    languages = []

    lang_text = get_section_text(text, [r'language', r'linguistic'])
    if not lang_text:
        # fallback: look for common "Languages: X, Y, Z" pattern anywhere
        lang_match = re.search(r'(?i)languages?\s*(?:known)?[\s:]+(.+?)(?:\n|$)', text)
        if lang_match:
            items = re.split(r'[,;|•]', lang_match.group(1))
            return [i.strip() for i in items if i.strip() and len(i.strip()) > 1]
        return languages

    lines = [l.strip() for l in lang_text.split('\n') if l.strip()]
    for line in lines:
        items = re.split(r'[,;|•]', line)
        for item in items:
            item = item.strip()
            if item and len(item) > 1 and len(item) < 30:
                languages.append(item)

    return languages


# ----------------------------------------------------------------
# HOBBIES / INTERESTS EXTRACTION
# ----------------------------------------------------------------

def extract_hobbies(text):
    """parses hobbies / interests / extracurricular activities"""
    hobbies = []

    hobby_text = get_section_text(text, [r'hobby|hobbies', r'interest', r'extracurricular', r'activities'])
    if not hobby_text:
        return hobbies

    lines = [l.strip() for l in hobby_text.split('\n') if l.strip()]
    for line in lines:
        items = re.split(r'[,;|•·●]', line)
        for item in items:
            item = item.strip(' -–')
            if item and len(item) > 2:
                hobbies.append(item)

    return hobbies


# ----------------------------------------------------------------
# TRAINING / WORKSHOPS EXTRACTION
# ----------------------------------------------------------------

def extract_trainings(text):
    """gets any training programs, workshops, or courses mentioned"""
    trainings = []

    train_text = get_section_text(text, [r'training', r'workshop', r'course', r'professional\s*development'])
    if not train_text:
        return trainings

    lines = [l.strip() for l in train_text.split('\n') if l.strip()]
    for line in lines:
        if len(line) > 3:
            trainings.append(line)

    return trainings


# ----------------------------------------------------------------
# TOTAL EXPERIENCE CALCULATOR
# ----------------------------------------------------------------

def calculate_total_experience(work_experience):
    """tries to figure out total years of experience from the durations"""
    from datetime import datetime

    total_months = 0
    month_map = {
        'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
        'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
    }

    for exp in work_experience:
        duration = exp.get("duration", "")
        if not duration:
            continue

        # try to parse "Month Year - Month Year" or "Year - Year"
        try:
            parts = re.split(r'[-–]|to', duration, maxsplit=1)
            if len(parts) == 2:
                end_str = parts[1].strip()

                # figure out end date
                if re.search(r'(?i)(present|current|ongoing|till\s*date)', end_str):
                    end_year = datetime.now().year
                    end_month = datetime.now().month
                else:
                    end_year_match = re.search(r'(20\d{2})', end_str)
                    end_month_match = re.search(r'(?i)(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)', end_str)
                    end_year = int(end_year_match.group(1)) if end_year_match else None
                    end_month = month_map.get(end_month_match.group(1).lower()[:3], 6) if end_month_match else 6

                # figure out start date
                start_str = parts[0].strip()
                start_year_match = re.search(r'(20\d{2}|19\d{2})', start_str)
                start_month_match = re.search(r'(?i)(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)', start_str)
                start_year = int(start_year_match.group(1)) if start_year_match else None
                start_month = month_map.get(start_month_match.group(1).lower()[:3], 1) if start_month_match else 1

                if start_year and end_year:
                    months = (end_year - start_year) * 12 + (end_month - start_month)
                    if 0 < months < 600:  # sanity check - cap at 50 years
                        total_months += months
        except Exception:
            continue

    if total_months > 0:
        years = total_months // 12
        months = total_months % 12
        if months > 0:
            return f"{years} year(s) {months} month(s)"
        return f"{years} year(s)"
    return None


print("[+] All extraction functions defined")

[+] All extraction functions defined


## Step 6: Main Parser Function

This ties everything together. Takes a file path, extracts text, runs BERT NER with proper chunking (respecting the 512 token limit), and calls each extraction function to build the final JSON.

In [6]:
def run_ner_chunked(text, pipe, tok, max_tokens=450):
    """
    runs the BERT NER model on text, splitting into token-safe chunks.

    BERT can only handle 512 tokens at once (including [CLS] and [SEP]),
    so we split by sentences and batch them into chunks of ~450 tokens.
    this avoids cutting words in half which would mess up entity detection.
    """
    all_entities = []

    # split into sentences - this gives us natural break points
    sentences = re.split(r'(?<=[.!?\n])\s+', text)

    current_chunk = ""
    for sent in sentences:
        # check if adding this sentence would overflow the token limit
        test = current_chunk + " " + sent if current_chunk else sent
        num_tokens = len(tok.encode(test, add_special_tokens=True))

        if num_tokens > max_tokens and current_chunk:
            # process what we have so far
            try:
                results = pipe(current_chunk)
                all_entities.extend(results)
            except Exception:
                pass
            current_chunk = sent
        else:
            current_chunk = test

    # process whatever is left in the buffer
    if current_chunk.strip():
        try:
            results = pipe(current_chunk)
            all_entities.extend(results)
        except Exception:
            pass

    return all_entities


def parse_resume(file_path):
    """
    the main deal - takes a resume file and spits out structured JSON.

    pipeline:
    1. extract raw text from file
    2. run BERT NER for entity recognition (on GPU if available)
    3. run each section extractor
    4. calculate derived fields (total experience etc.)
    5. package everything into a clean JSON structure
    """
    print(f"\n{'='*60}")
    print(f"  RESUME PARSER")
    print(f"  File: {os.path.basename(file_path)}")
    print(f"  Device: {'GPU' if device == 0 else 'CPU'}")
    print(f"{'='*60}")

    # -- step 1: get the text out of the file --
    print("\n[1/5] Extracting text...")
    raw_text = extract_text(file_path)
    word_count = len(raw_text.split())
    print(f"      {len(raw_text)} chars, {word_count} words extracted")

    # -- step 2: run BERT NER --
    print("[2/5] Running BERT NER model...")
    ner_results = run_ner_chunked(raw_text, ner_pipeline, tokenizer)
    # only keep high-confidence predictions
    ner_results_filtered = [e for e in ner_results if e["score"] > 0.70]
    print(f"      {len(ner_results)} entities found ({len(ner_results_filtered)} above 70% confidence)")

    # -- step 3: extract each section --
    print("[3/5] Parsing sections...")
    contact = extract_contact_info(raw_text, ner_results)
    education = extract_education(raw_text)
    experience = extract_work_experience(raw_text)
    skills = extract_skills(raw_text)
    certifications = extract_certifications(raw_text)
    projects = extract_projects(raw_text)
    objective = extract_objective(raw_text)
    achievements = extract_achievements(raw_text)
    languages = extract_languages(raw_text)
    hobbies = extract_hobbies(raw_text)
    trainings = extract_trainings(raw_text)

    # -- step 4: derived fields --
    print("[4/5] Computing derived fields...")
    total_exp = calculate_total_experience(experience)

    # get all orgs and locations detected by transformer
    detected_orgs = list(set([
        e["word"].replace("##", "").strip()
        for e in ner_results if e["entity_group"] == "ORG" and e["score"] > 0.75
    ]))
    detected_locs = list(set([
        e["word"].replace("##", "").strip()
        for e in ner_results if e["entity_group"] == "LOC" and e["score"] > 0.75
    ]))
    detected_persons = list(set([
        e["word"].replace("##", "").strip()
        for e in ner_results if e["entity_group"] == "PER" and e["score"] > 0.75
    ]))

    # -- step 5: build the output JSON --
    print("[5/5] Building JSON output...")

    output = OrderedDict([
        ("contact_information", OrderedDict([
            ("name", contact.get("name")),
            ("email", contact.get("email")),
            ("phone", contact.get("phone")),
            ("linkedin", contact.get("linkedin")),
            ("github", contact.get("github")),
            ("portfolio", contact.get("portfolio")),
            ("location", contact.get("location")),
        ])),
        ("objective_or_summary", objective),
        ("total_experience", total_exp),
        ("education", education),
        ("work_experience", experience),
        ("skills", skills),
        ("projects", projects),
        ("certifications", certifications),
        ("achievements", achievements),
        ("languages", languages),
        ("trainings_and_workshops", trainings),
        ("hobbies_and_interests", hobbies),
        ("ner_detected_entities", OrderedDict([
            ("persons", detected_persons),
            ("organizations", detected_orgs),
            ("locations", detected_locs),
        ])),
        ("metadata", OrderedDict([
            ("source_file", os.path.basename(file_path)),
            ("file_format", os.path.splitext(file_path)[1]),
            ("word_count", word_count),
            ("model_used", model_name),
            ("device_used", "GPU (CUDA)" if device == 0 else "CPU"),
            ("total_entities_detected", len(ner_results)),
        ])),
    ])

    # quick summary of what we found
    print(f"\n{'─'*40}")
    print(f"  Contact fields: {sum(1 for v in contact.values() if v)}/7")
    print(f"  Education entries: {len(education)}")
    print(f"  Experience entries: {len(experience)}")
    print(f"  Skills: {sum(len(v) for v in skills.values()) if isinstance(skills, dict) else len(skills)}")
    print(f"  Projects: {len(projects)}")
    print(f"  Certifications: {len(certifications)}")
    print(f"  Achievements: {len(achievements)}")
    print(f"  Languages: {len(languages)}")
    print(f"  Total Experience: {total_exp or 'N/A'}")
    print(f"{'─'*40}")
    print(f"  [DONE]")
    print(f"{'='*60}")

    return output


print("[+] Main parser function ready")

[+] Main parser function ready


## Step 7: Output Utilities

In [7]:
def save_to_json(data, output_path="parsed_resume.json"):
    """dumps the parsed data to a json file"""
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    file_size = os.path.getsize(output_path)
    print(f"[+] Saved to: {output_path} ({file_size} bytes)")
    return output_path


def display_json(data):
    """prints the json in a nice readable format"""
    output = json.dumps(data, indent=4, ensure_ascii=False)
    print(output)
    return output


print("[+] Output utilities ready")

[+] Output utilities ready


## Step 8: Parse Resume

Provide the path to your resume file below (.pdf, .docx, or .doc) and run the cell. The output JSON will be saved with a filename based on the input file.

In [9]:
# put the path to your resume file here
resume_file = "/content/TF59ab7c0a-01b4-4a8b-8704-1629ef74476b217f4526_wac-e5cb2f7654bf.docx"

# parse it
result = parse_resume(resume_file)

# generate output filename dynamically from the input file name
# e.g. "Gaurav_Kumar_resume.pdf" -> "Gaurav_Kumar_resume_parsed.json"
base_name = os.path.splitext(os.path.basename(resume_file))[0]
output_json = f"{base_name}_parsed.json"

# display the full parsed output
print("\n" + "=" * 60)
print("  PARSED RESUME - JSON OUTPUT")
print("=" * 60 + "\n")
display_json(result)

# save to json
save_to_json(result, output_json)


  RESUME PARSER
  File: TF59ab7c0a-01b4-4a8b-8704-1629ef74476b217f4526_wac-e5cb2f7654bf.docx
  Device: CPU

[1/5] Extracting text...
      923 chars, 113 words extracted
[2/5] Running BERT NER model...
      8 entities found (8 above 70% confidence)
[3/5] Parsing sections...
[4/5] Computing derived fields...
[5/5] Building JSON output...

────────────────────────────────────────
  Contact fields: 3/7
  Education entries: 1
  Experience entries: 1
  Skills: 6
  Projects: 0
  Certifications: 0
  Achievements: 3
  Languages: 0
  Total Experience: N/A
────────────────────────────────────────
  [DONE]

  PARSED RESUME - JSON OUTPUT

{
    "contact_information": {
        "name": "Alexander Martens",
        "email": "alex@lamnahealthcare.com",
        "phone": "678-555-0103",
        "linkedin": null,
        "github": null,
        "portfolio": null,
        "location": null
    },
    "objective_or_summary": null,
    "total_experience": null,
    "education": [
        {
            "in

'TF59ab7c0a-01b4-4a8b-8704-1629ef74476b217f4526_wac-e5cb2f7654bf_parsed.json'

## Output Schema Reference

```json
{
    "contact_information": {
        "name": "Full Name",
        "email": "email@domain.com",
        "phone": "+91 XXXXX XXXXX",
        "linkedin": "linkedin.com/in/...",
        "github": "github.com/...",
        "portfolio": "website URL",
        "location": "City, State"
    },
    "objective_or_summary": "Career objective or professional summary text",
    "total_experience": "X year(s) Y month(s)",
    "education": [
        {
            "degree": "B.Tech in Computer Science",
            "institution": "University Name",
            "graduation_year": "2023",
            "start_year": "2019",
            "cgpa": "8.5",
            "cgpa_scale": "10",
            "percentage": "85%",
            "specialization": "Machine Learning"
        }
    ],
    "work_experience": [
        {
            "company": "Company Name",
            "position": "Software Engineer",
            "duration": "Jan 2023 - Present",
            "employment_type": "Full-time",
            "description": "Responsibilities and achievements"
        }
    ],
    "skills": {
        "technical_skills": ["Python", "Java", "SQL"],
        "tools": ["Docker", "Git", "AWS"],
        "frameworks": ["React", "Django", "TensorFlow"]
    },
    "projects": [
        {
            "title": "Project Name",
            "description": "What the project does",
            "tech_stack": "Python, FastAPI, React",
            "link": "github.com/..."
        }
    ],
    "certifications": [
        {"name": "AWS Solutions Architect", "year": "2023"}
    ],
    "achievements": ["Award or accomplishment"],
    "languages": ["English", "Hindi"],
    "trainings_and_workshops": ["Training program details"],
    "hobbies_and_interests": ["Reading", "Coding"],
    "ner_detected_entities": {
        "persons": ["Names detected by BERT"],
        "organizations": ["Companies/Universities detected"],
        "locations": ["Cities/Countries detected"]
    },
    "metadata": {
        "source_file": "filename.pdf",
        "file_format": ".pdf",
        "word_count": 500,
        "model_used": "dslim/bert-base-NER",
        "device_used": "GPU (CUDA)",
        "total_entities_detected": 42
    }
}
```

**Technical Details:**
- Model: `dslim/bert-base-NER` (BERT-base, 110M params, fine-tuned on CoNLL-2003)
- Entity types: PER, ORG, LOC, MISC
- Token limit: 512 per chunk, using sentence-aware splitting at 450 tokens
- Device: CUDA GPU when available, falls back to CPU
- Aggregation: Simple (merges WordPiece subword tokens into full entities)
- Backup NER: spaCy en_core_web_sm for fallback name detection